# S002 - Unchanged T006 checkpoint inference

The user approved one competition submission using the saved epoch-1 T006 vector-attention checkpoint. Do not retrain or refit this model.

Run in the pinned Kaggle training image with the ShareChat competition and T006 kernel outputs attached. Verify input hashes and package versions, replay validation AUC, and only then export test predictions. All definitions are frozen below; training functions are defined but never called. The only invoked main function is the inference runner.

**This notebook never submits to the competition.** The one approved submission is a separate CLI action after output verification. See README.md and provenance.json for the immutable record.

In [ ]:
import hashlib
import sys
from pathlib import Path

if not Path("/kaggle/input").is_dir():
    raise RuntimeError("Run this submission notebook on Kaggle in the original training image")
__file__ = str(Path.cwd() / "predict_custom.py")
data_candidates = [
    path.parent for path in Path("/kaggle/input").rglob("train.parquet")
    if (path.parent / "test.parquet").is_file()
    and (path.parent / "sample_submission.csv").is_file()
]
model_candidates = [
    path.parent.parent for path in Path("/kaggle/input").rglob("best_model.pt")
    if path.parent.name == "attention"
    and (path.parent.parent / "preprocessing.joblib").is_file()
    and (path.parent.parent / "results.json").is_file()
]
if len(data_candidates) != 1 or len(model_candidates) != 1:
    raise RuntimeError(f"Expected one competition dataset and one T006 output: {data_candidates}, {model_candidates}")
DATA, MODEL = data_candidates[0], model_candidates[0]
OUTPUT = Path("/kaggle/working")
EXPECTED = {
    "attention/best_model.pt": "454a93210a7ae5408fbaf92828e74e010f26f041ace3939d8fde3a10ebd74f3f",
    "preprocessing.joblib": "9a7a431961913de4a6e29cd8c21a855c3333560f7d3ea53a501a0377a9587983",
    "results.json": "2c4d911a2f992aa69cae2357373adb326d5f457e7d7ed90c96913575df63d09e"
}
for relative, expected in EXPECTED.items():
    with (MODEL / relative).open("rb") as stream:
        actual = hashlib.file_digest(stream, "sha256").hexdigest()
    if actual != expected:
        raise ValueError(f"T006 artifact checksum mismatch: {relative}")
print("Original T006 checkpoint, preprocessor, and results checksums verified")
sys.argv = [
    __file__, "--model-dir", str(MODEL), "--data-dir", str(DATA),
    "--output-dir", str(OUTPUT), "--device", "cuda",
]


In [ ]:
import argparse
import gc
import json
from pathlib import Path
import time

from catboost import CatBoostClassifier
from catboost.utils import get_gpu_device_count
import numpy as np
import polars as pl
import pyarrow.parquet as pq
from sklearn.metrics import roc_auc_score


ROOT = Path(__file__).resolve().parent
ON_KAGGLE = Path("/kaggle/input").is_dir()
TARGET = "watched"
TIME = "feature1"


def features(frame, expand_embeddings=False):
    expressions = []
    for name, dtype in frame.schema.items():
        if name in {TARGET, "request_id", "id"}:
            continue
        column = pl.col(name)
        if isinstance(dtype, pl.Datetime):
            expressions.extend(
                [
                    column.dt.hour().cast(pl.Float32).alias(f"{name}_hour"),
                    column.dt.weekday().cast(pl.Float32).alias(f"{name}_weekday"),
                ]
            )
        elif isinstance(dtype, pl.List):
            expressions.extend(
                [
                    column.list.len().cast(pl.Float32).alias(f"{name}_length"),
                    column.list.mean().cast(pl.Float32).alias(f"{name}_mean"),
                    column.list.std().cast(pl.Float32).alias(f"{name}_std"),
                    column.list.min().cast(pl.Float32).alias(f"{name}_min"),
                    column.list.max().cast(pl.Float32).alias(f"{name}_max"),
                ]
            )
            if expand_embeddings:
                invalid = frame.select(
                    (column.is_not_null() & (column.list.len() != 32)).any()
                ).item()
                if invalid:
                    raise ValueError(f"{name} must contain 32-element vectors")
                expressions.extend(
                    column.list.get(index).cast(pl.Float32).alias(f"{name}_dim_{index}")
                    for index in range(32)
                )
        elif dtype == pl.String:
            expressions.append(column.fill_null("__MISSING__").alias(name))
        elif dtype.is_numeric():
            expressions.append(column.cast(pl.Float32).alias(name))
        else:
            raise TypeError(f"Unsupported feature {name}: {dtype}")
    return frame.select(expressions).with_columns(
        pl.col(pl.Float32).fill_nan(None)
    )


def load_sample(path, maximum_rows, seed, expand_embeddings=False):
    parquet = pq.ParquetFile(path)
    total = parquet.metadata.num_rows
    rng = np.random.default_rng(seed)
    indices = np.sort(rng.choice(total, min(maximum_rows, total), replace=False))
    frames = []
    offset = 0
    for batch in parquet.iter_batches(batch_size=8192):
        end = offset + batch.num_rows
        selected = indices[(indices >= offset) & (indices < end)] - offset
        if len(selected):
            raw = pl.from_arrow(batch.take(selected))
            frames.append(
                features(raw, expand_embeddings).with_columns(
                    raw[TARGET], raw[TIME].alias("_time")
                )
            )
        offset = end
    return pl.concat(frames)


def model(iterations, threads, task_type):
    return CatBoostClassifier(
        iterations=iterations,
        depth=6,
        learning_rate=0.06,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        thread_count=threads,
        task_type=task_type,
        max_ctr_complexity=1,
        allow_writing_files=False,
        verbose=50,
    )


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--rows", type=int, default=1_600_000 if ON_KAGGLE else 150_000)
    parser.add_argument("--iterations", type=int, default=400)
    parser.add_argument("--threads", type=int, default=4)
    parser.add_argument("--expand-embeddings", action="store_true")
    parser.add_argument("--validation-only", action="store_true")
    parser.add_argument(
        "--task-type", choices=["CPU", "GPU"], default="GPU" if ON_KAGGLE else "CPU"
    )
    parser.add_argument("--data-dir", type=Path)
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("/kaggle/working") if ON_KAGGLE else ROOT / "artifacts",
    )
    args = parser.parse_args()
    if min(args.rows, args.iterations, args.threads) <= 0:
        parser.error("rows, iterations, and threads must be positive")

    if args.task_type == "GPU" and get_gpu_device_count() == 0:
        raise RuntimeError("GPU training requested, but no CUDA GPU is available")
    data = args.data_dir
    if data is None and ON_KAGGLE:
        candidates = [
            path.parent
            for path in Path("/kaggle/input").rglob("train.parquet")
            if (path.parent / "test.parquet").is_file()
            and (path.parent / "sample_submission.csv").is_file()
        ]
        if len(candidates) != 1:
            raise RuntimeError(f"Expected one ShareChat dataset, found {candidates}")
        data = candidates[0]
    elif data is None:
        data = ROOT / "data"
    output = args.output_dir
    output.mkdir(parents=True, exist_ok=True)
    print(f"Compute: {args.task_type}; input: {data}; output: {output}", flush=True)
    sample = load_sample(
        data / "train.parquet", args.rows, 42, args.expand_embeddings
    ).sort("_time")
    if sample.height < 2:
        raise ValueError("Chronological validation requires at least two rows")
    if sample["_time"].null_count():
        raise ValueError("Missing timestamps prevent chronological validation")
    cutoff = sample["_time"][int(sample.height * 0.8)]
    train = sample.filter(pl.col("_time") < cutoff)
    valid = sample.filter(pl.col("_time") >= cutoff)
    if train.is_empty() or valid.is_empty():
        raise ValueError(
            f"Timestamp cutoff {cutoff} cannot produce nonempty chronological splits"
        )
    for name, part in [("train", train), ("validation", valid)]:
        if part[TARGET].n_unique() != 2:
            raise ValueError(f"{name} must contain both target classes")
    columns = [name for name in sample.columns if name not in {TARGET, "_time"}]
    categories = [name for name in columns if sample.schema[name] == pl.String]
    x_train = train.select(columns).to_pandas()
    x_valid = valid.select(columns).to_pandas()
    y_train = train[TARGET].to_numpy()
    y_valid = valid[TARGET].to_numpy()
    sample_rows, training_rows, validation_rows = sample.height, train.height, valid.height
    print(
        f"Training: {train.height:,}; validation: {valid.height:,}; "
        f"features: {len(columns)}; cutoff: {cutoff}",
        flush=True,
    )
    if args.validation_only:
        del sample, train, valid
        gc.collect()
    estimator = model(args.iterations, args.threads, args.task_type)
    fit_started = time.monotonic()
    estimator.fit(
        x_train,
        y_train,
        cat_features=categories,
        eval_set=(x_valid, y_valid),
        early_stopping_rounds=60,
    )
    fit_seconds = time.monotonic() - fit_started
    auc = roc_auc_score(
        y_valid, estimator.predict_proba(x_valid)[:, 1]
    )
    best_iterations = estimator.tree_count_
    metrics = {
        "validation_auc": auc,
        "validation": "chronological 80/20 split; timestamp ties kept together",
        "cutoff": str(cutoff),
        "sample_rows": sample_rows,
        "training_rows": training_rows,
        "validation_rows": validation_rows,
        "iterations": best_iterations,
        "max_iterations": args.iterations,
        "expand_embeddings": args.expand_embeddings,
        "validation_only": args.validation_only,
        "fit_seconds": fit_seconds,
        "task_type": args.task_type,
        "features": columns,
        "categorical_features": categories,
    }
    estimator.save_model(str(output / "validation.cbm"))
    (output / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
    print(f"Validation ROC AUC: {auc:.6f}", flush=True)
    if args.validation_only:
        print("Validation-only run: skipping full-data refit and test predictions", flush=True)
        return
    del estimator, x_train, x_valid, train, valid
    gc.collect()

    print("Refitting on the full sample", flush=True)
    estimator = model(best_iterations, args.threads, args.task_type)
    estimator.fit(
        sample.select(columns).to_pandas(),
        sample[TARGET].to_numpy(),
        cat_features=categories,
    )
    estimator.save_model(str(output / "baseline.cbm"))
    del sample
    gc.collect()

    submission = pl.read_csv(data / "sample_submission.csv")
    if submission.columns != ["id", TARGET]:
        raise ValueError(f"Unexpected submission schema: {submission.columns}")
    test = pq.ParquetFile(data / "test.parquet")
    if test.metadata.num_rows != submission.height:
        raise ValueError("Test and submission row counts differ")
    predictions = []
    offset = 0
    for batch in test.iter_batches(batch_size=8192):
        raw = pl.from_arrow(batch)
        if "id" not in raw.columns:
            raise ValueError("Test data has no id column for submission alignment")
        expected = submission["id"].slice(offset, raw.height)
        if raw["id"].to_list() != expected.to_list():
            raise ValueError("Test ids do not match sample submission order")
        predictions.append(
            estimator.predict_proba(
                features(raw, args.expand_embeddings).select(columns).to_pandas()
            )[:, 1]
        )
        offset += raw.height
    probabilities = np.concatenate(predictions)
    if not np.isfinite(probabilities).all() or not (
        (probabilities >= 0) & (probabilities <= 1)
    ).all():
        raise ValueError("Invalid predicted probabilities")
    submission.with_columns(pl.Series(TARGET, probabilities)).write_csv(
        output / "submission.csv"
    )
    print(f"Saved {submission.height:,} predictions to {output / 'submission.csv'}")




In [ ]:
import argparse
import gc
import hashlib
from importlib.metadata import version
import json
from pathlib import Path
import time

import joblib
import numpy as np
import polars as pl
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
import torch
from torch import nn
from torch.nn import functional as F



def prepare_features(train, valid):
    columns = [name for name in train.columns if name not in {TARGET, "_time"}]
    numeric = [name for name in columns if train.schema[name].is_numeric()]
    categorical = [name for name in columns if train.schema[name] == pl.String]
    if len(numeric) + len(categorical) != len(columns):
        raise TypeError("The engineered sample must contain only numeric/string features")
    imputer = SimpleImputer(
        strategy="median", keep_empty_features=True, add_indicator=True, copy=False
    )
    scaler = StandardScaler(copy=False)
    transformed, infinity_counts = [], []
    for label, frame in [("training", train), ("validation", valid)]:
        array = np.array(frame.select(numeric).to_numpy(), dtype=np.float32, order="C")
        count = int(np.isinf(array).sum())
        infinity_counts.append(count)
        if count:
            print(f"Treating {count} infinite {label} values as missing", flush=True)
            array[np.isinf(array)] = np.nan
        if label == "training":
            array = imputer.fit_transform(array)
            array = scaler.fit_transform(array)
        else:
            array = imputer.transform(array)
            array = scaler.transform(array)
        transformed.append(array)
    x_train, x_valid = transformed
    for array in [x_train, x_valid]:
        np.clip(array, -10, 10, out=array)
        if not np.isfinite(array).all():
            raise ValueError("Nonfinite numerical features after preprocessing")
    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value", unknown_value=-1, dtype=np.int64
    )
    categories = [
        frame.select(categorical).fill_null("__MISSING__").to_numpy()
        for frame in [train, valid]
    ]
    c_train = encoder.fit_transform(categories[0]) + 1
    c_valid = encoder.transform(categories[1]) + 1
    cardinalities = [len(values) + 1 for values in encoder.categories_]
    preprocessing = {
        "numeric_columns": numeric,
        "categorical_columns": categorical,
        "imputer": imputer,
        "scaler": scaler,
        "encoder": encoder,
        "cardinalities": cardinalities,
        "infinity_counts": infinity_counts,
        "validation_unknown_category_rates": {
            name: float((c_valid[:, index] == 0).mean())
            for index, name in enumerate(categorical)
        },
    }
    return (
        torch.from_numpy(np.ascontiguousarray(x_train, dtype=np.float32)),
        torch.from_numpy(np.ascontiguousarray(c_train)),
        torch.from_numpy(np.ascontiguousarray(x_valid, dtype=np.float32)),
        torch.from_numpy(np.ascontiguousarray(c_valid)),
        preprocessing,
    )


def transform_features(frame, state):
    numeric = np.array(
        frame.select(state["numeric_columns"]).to_numpy(),
        dtype=np.float32,
        order="C",
    )
    infinities = np.isinf(numeric)
    if infinities.any():
        print(f"Treating {int(infinities.sum())} infinite inference values as missing", flush=True)
        numeric[infinities] = np.nan
    numeric = state["scaler"].transform(state["imputer"].transform(numeric))
    np.clip(numeric, -10, 10, out=numeric)
    if not np.isfinite(numeric).all():
        raise ValueError("Nonfinite numerical features after inference preprocessing")
    categories = (
        frame.select(state["categorical_columns"]).fill_null("__MISSING__").to_numpy()
    )
    encoded = state["encoder"].transform(categories) + 1
    return (
        torch.from_numpy(np.ascontiguousarray(numeric, dtype=np.float32)),
        torch.from_numpy(np.ascontiguousarray(encoded, dtype=np.int64)),
    )


class TabMClassifier(nn.Module):
    def __init__(
        self, n_numeric, cardinalities, members=8, width=128, blocks=2, dropout=0.1
    ):
        super().__init__()
        from tabm import TabM

        self.embeddings = nn.ModuleList(
            nn.Embedding(cardinality, 8, padding_idx=0)
            for cardinality in cardinalities
        )
        for embedding in self.embeddings:
            nn.init.normal_(embedding.weight, std=0.02)
            with torch.no_grad():
                embedding.weight[0].zero_()
        self.backbone = TabM.make(
            n_num_features=n_numeric + 8 * len(cardinalities),
            d_out=1,
            k=members,
            n_blocks=blocks,
            d_block=width,
            dropout=dropout,
        )

    def forward(self, numeric, categorical):
        embedded = [
            layer(categorical[:, index])
            for index, layer in enumerate(self.embeddings)
        ]
        return self.backbone(torch.cat([numeric, *embedded], dim=1)).squeeze(-1)


@torch.inference_mode()
def predict(model, numeric, categorical, device, batch_size):
    model.eval()
    probabilities = []
    for start in range(0, len(numeric), batch_size):
        logits = model(
            numeric[start : start + batch_size].to(device),
            categorical[start : start + batch_size].to(device),
        )
        probabilities.append(logits.sigmoid().mean(dim=1).cpu().numpy())
    return np.concatenate(probabilities)


def fit_model(
    model, model_config, optimizer, x_train, c_train, y_train, x_valid, c_valid, y_valid,
    *, device, output, epochs, patience, batch_size,
):
    generator = torch.Generator().manual_seed(42)
    best_auc, best_epoch, stale = -np.inf, 0, 0
    history = []
    training_started = time.monotonic()
    for epoch in range(1, epochs + 1):
        epoch_started = time.monotonic()
        model.train()
        order = torch.randperm(len(y_train), generator=generator)
        loss_sum = 0.0
        for indices in order.split(batch_size):
            optimizer.zero_grad(set_to_none=True)
            logits = model(x_train[indices].to(device), c_train[indices].to(device))
            targets = y_train[indices].to(device).unsqueeze(1).expand_as(logits)
            loss = F.binary_cross_entropy_with_logits(logits, targets)
            if not torch.isfinite(loss).item():
                raise ValueError(f"Nonfinite training loss at epoch {epoch}")
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(indices)
        probabilities = predict(model, x_valid, c_valid, device, batch_size)
        auc = float(roc_auc_score(y_valid, probabilities))
        record = {
            "epoch": epoch,
            "training_loss": loss_sum / len(y_train),
            "validation_auc": auc,
            "seconds": time.monotonic() - epoch_started,
        }
        history.append(record)
        print(json.dumps(record), flush=True)
        if auc > best_auc:
            best_auc, best_epoch, stale = auc, epoch, 0
            torch.save(
                {
                    "model_config": model_config,
                    "state_dict": {
                        key: value.detach().cpu().clone()
                        for key, value in model.state_dict().items()
                    },
                },
                output / "best_model.pt",
            )
        else:
            stale += 1
            if stale >= patience:
                break
    training_seconds = time.monotonic() - training_started
    checkpoint = torch.load(output / "best_model.pt", map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["state_dict"])
    probabilities = predict(model, x_valid, c_valid, device, batch_size)
    restored_auc = float(roc_auc_score(y_valid, probabilities))
    if abs(restored_auc - best_auc) > 1e-5:
        raise ValueError("Reloaded checkpoint does not reproduce the selected validation AUC")
    np.savez_compressed(
        output / "validation_predictions.npz", y_true=y_valid, probabilities=probabilities
    )
    return {
        "validation_auc": restored_auc,
        "best_epoch": best_epoch,
        "epochs_run": len(history),
        "training_seconds": training_seconds,
        "history": history,
    }


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--rows", type=int, default=150_000)
    parser.add_argument("--sample-path", type=Path)
    parser.add_argument("--epochs", type=int, default=20)
    parser.add_argument("--patience", type=int, default=5)
    parser.add_argument("--batch-size", type=int, default=1024)
    parser.add_argument("--members", type=int, default=8)
    parser.add_argument("--width", type=int, default=128)
    parser.add_argument("--blocks", type=int, default=2)
    parser.add_argument("--threads", type=int, default=4)
    parser.add_argument("--device", choices=["auto", "cpu", "mps", "cuda"], default="auto")
    parser.add_argument("--output-dir", type=Path, default=ROOT / "artifacts/local-tabm-v1")
    args = parser.parse_args()
    if min(
        args.rows, args.epochs, args.patience, args.batch_size,
        args.members, args.width, args.blocks, args.threads,
    ) <= 0:
        parser.error("All numerical arguments must be positive")
    device = args.device
    if device == "auto":
        device = (
            "cuda" if torch.cuda.is_available()
            else "mps" if torch.backends.mps.is_available()
            else "cpu"
        )
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but unavailable")
    if device == "mps" and not torch.backends.mps.is_available():
        raise RuntimeError("Apple MPS requested but unavailable")
    torch.set_num_threads(args.threads)
    torch.manual_seed(42)
    started = time.monotonic()
    sample_hash = None
    if args.sample_path:
        sample = pl.read_parquet(args.sample_path)
        if sample.height != args.rows:
            raise ValueError("Cached sample row count does not match --rows")
        with args.sample_path.open("rb") as stream:
            sample_hash = hashlib.file_digest(stream, "sha256").hexdigest()
    else:
        sample = load_sample(ROOT / "data/train.parquet", args.rows, 42)
    sample = sample.select(
        name for name in sample.columns if "_dim_" not in name
    ).sort("_time")
    if sample.height < 2 or sample["_time"].null_count():
        raise ValueError("Chronological validation requires nonmissing timestamps")
    cutoff = sample["_time"][int(sample.height * 0.8)]
    train = sample.filter(pl.col("_time") < cutoff)
    valid = sample.filter(pl.col("_time") >= cutoff)
    for label, frame in [("training", train), ("validation", valid)]:
        if set(frame[TARGET].unique().to_list()) != {0, 1}:
            raise ValueError(f"{label} must contain both binary target classes")
    training_rows, validation_rows = train.height, valid.height
    y_train = torch.tensor(train[TARGET].to_numpy().copy(), dtype=torch.float32)
    y_valid = valid[TARGET].to_numpy().copy()
    x_train, c_train, x_valid, c_valid, preprocessing = prepare_features(train, valid)
    feature_count = sample.width - 2
    sample_rows = sample.height
    del sample, train, valid
    gc.collect()
    output = args.output_dir
    output.mkdir(parents=True, exist_ok=True)
    joblib.dump(preprocessing, output / "preprocessing.joblib")
    model_config = {
        "n_numeric": x_train.shape[1],
        "cardinalities": preprocessing["cardinalities"],
        "members": args.members,
        "width": args.width,
        "blocks": args.blocks,
        "dropout": 0.1,
    }
    model = TabMClassifier(**model_config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.002, weight_decay=0.0003)
    preparation_seconds = time.monotonic() - started
    print(
        f"Device: {device}; sample: {sample_rows:,}; train: {training_rows:,}; "
        f"valid: {validation_rows:,}; features: {feature_count}; "
        f"cutoff: {cutoff}; preparation: {preparation_seconds:.1f}s",
        flush=True,
    )
    fitted = fit_model(
        model, model_config, optimizer, x_train, c_train, y_train, x_valid, c_valid, y_valid,
        device=device, output=output, epochs=args.epochs, patience=args.patience,
        batch_size=args.batch_size,
    )
    report = {
        "algorithm": "TabM with learned categorical embeddings",
        **fitted,
        "seed": 42,
        "sample_rows": sample_rows,
        "sample_sha256": sample_hash,
        "training_rows": training_rows,
        "validation_rows": validation_rows,
        "cutoff": str(cutoff),
        "input_features": feature_count,
        "numerical_features": len(preprocessing["numeric_columns"]),
        "numerical_features_after_imputation": x_train.shape[1],
        "categorical_features": len(preprocessing["categorical_columns"]),
        "validation_unknown_category_rates": preprocessing["validation_unknown_category_rates"],
        "infinite_input_counts_train_valid": preprocessing["infinity_counts"],
        "model_config": model_config,
        "optimizer": {"name": "AdamW", "learning_rate": 0.002, "weight_decay": 0.0003},
        "max_epochs": args.epochs,
        "patience": args.patience,
        "batch_size": args.batch_size,
        "device": device,
        "threads": args.threads,
        "preparation_seconds": preparation_seconds,
        "versions": {
            name: version(name)
            for name in ["torch", "tabm", "numpy", "polars", "scikit-learn"]
        },
        "competition_submission": False,
    }
    (output / "results.json").write_text(json.dumps(report, indent=2) + "\n")
    print(
        f"Best validation ROC AUC: {fitted['validation_auc']:.6f}; "
        f"epoch: {fitted['best_epoch']}; training: {fitted['training_seconds']:.1f}s; "
        "no submission performed",
        flush=True,
    )




In [ ]:
import argparse
import gc
import hashlib
from importlib.metadata import version
import json
from pathlib import Path
import time

import joblib
import numpy as np
import polars as pl
import torch
from torch import nn
from torch.nn import functional as F



def pack_vectors(frame, fields):
    packed, missing = [], {}
    for field in fields:
        columns = [f"{field}_dim_{index}" for index in range(32)]
        raw = np.array(frame.select(columns).to_numpy(), dtype=np.float32, order="C")
        valid = np.isfinite(raw).all(axis=1)
        missing[field] = int((~valid).sum())
        raw[~valid] = 0
        norm = np.linalg.norm(raw.astype(np.float64), axis=1)
        unit = (raw / np.maximum(norm, 1e-8)[:, None]).astype(np.float32)
        log_norm = np.clip(np.log1p(norm), 0, 20).astype(np.float32)
        packed.append(
            np.concatenate([unit, log_norm[:, None], valid[:, None]], axis=1)
        )
    return torch.from_numpy(np.stack(packed, axis=1).astype(np.float32)), missing


def append_vectors(frame, numeric, fields):
    packed = torch.empty(
        (len(numeric), numeric.shape[1] + 34 * len(fields)), dtype=torch.float32
    )
    packed[:, : numeric.shape[1]] = numeric
    missing = {}
    for index, field in enumerate(fields):
        vector, counts = pack_vectors(frame, [field])
        start = numeric.shape[1] + 34 * index
        packed[:, start : start + 34] = vector[:, 0]
        missing.update(counts)
    return packed, missing


def prepare_custom_features(train, valid, min_category_count):
    dimension_columns = [name for name in train.columns if "_dim_" in name]
    fields = sorted(
        {name.rsplit("_dim_", 1)[0] for name in dimension_columns},
        key=lambda name: int(name.removeprefix("feature")),
    )
    if not fields or set(dimension_columns) != {
        f"{field}_dim_{index}" for field in fields for index in range(32)
    }:
        raise ValueError("Expected complete 32-dimensional vector fields")
    summary = [name for name in train.columns if "_dim_" not in name]
    x_train, c_train, x_valid, c_valid, state = prepare_features(
        train.select(summary), valid.select(summary)
    )
    mappings, cardinalities = [], []
    mapped_train = np.empty(c_train.shape, dtype=np.int64)
    mapped_valid = np.empty(c_valid.shape, dtype=np.int64)
    for index, cardinality in enumerate(state["cardinalities"]):
        counts = np.bincount(c_train[:, index].numpy(), minlength=cardinality)
        retained = np.flatnonzero(
            (counts >= min_category_count) & (np.arange(cardinality) != 0)
        )
        mapping = np.zeros(cardinality, dtype=np.int64)
        mapping[retained] = np.arange(1, len(retained) + 1)
        mapped_train[:, index] = mapping[c_train[:, index].numpy()]
        mapped_valid[:, index] = mapping[c_valid[:, index].numpy()]
        mappings.append(mapping)
        cardinalities.append(len(retained) + 1)
    n_dense = x_train.shape[1]
    x_train, missing_train = append_vectors(train, x_train, fields)
    x_valid, missing_valid = append_vectors(valid, x_valid, fields)
    state.update(
        {
            "original_cardinalities": state["cardinalities"],
            "cardinalities": cardinalities,
            "category_remappings": mappings,
            "min_category_count": min_category_count,
            "vector_fields": fields,
            "n_dense": n_dense,
            "missing_vectors_train": missing_train,
            "missing_vectors_valid": missing_valid,
            "validation_unknown_or_rare_category_rates": {
                name: float((mapped_valid[:, index] == 0).mean())
                for index, name in enumerate(state["categorical_columns"])
            },
        }
    )
    return (
        x_train, torch.from_numpy(mapped_train),
        x_valid, torch.from_numpy(mapped_valid), state,
    )


def transform_custom_features(frame, state):
    numeric, categorical = transform_features(frame, state)
    remapped = np.empty(categorical.shape, dtype=np.int64)
    for index, mapping in enumerate(state["category_remappings"]):
        remapped[:, index] = mapping[categorical[:, index].numpy()]
    numeric, _ = append_vectors(frame, numeric, state["vector_fields"])
    return numeric, torch.from_numpy(remapped)


class RecommendationNetwork(nn.Module):
    def __init__(
        self, n_dense, cardinalities, n_vectors, attention=False, positive_rate=0.05
    ):
        super().__init__()
        self.n_dense = n_dense
        self.n_vectors = n_vectors
        self.embeddings = nn.ModuleList(
            nn.Embedding(cardinality, 8, padding_idx=0)
            for cardinality in cardinalities
        )
        for embedding in self.embeddings:
            nn.init.normal_(embedding.weight, std=0.02)
            with torch.no_grad():
                embedding.weight[0].zero_()
        self.scalar_branch = nn.Sequential(
            nn.Linear(n_dense, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.2)
        )
        self.category_branch = nn.Sequential(
            nn.Linear(8 * len(cardinalities), 64),
            nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.3),
        )
        self.vector_weights = nn.Parameter(torch.empty(n_vectors, 33, 32))
        for weight in self.vector_weights:
            nn.init.xavier_uniform_(weight)
        self.vector_bias = nn.Parameter(torch.zeros(n_vectors, 32))
        self.missing_tokens = nn.Parameter(torch.randn(n_vectors, 32) * 0.02)
        self.field_tokens = nn.Parameter(torch.randn(n_vectors, 32) * 0.02)
        self.vector_transform = nn.Sequential(
            nn.LayerNorm(32), nn.Linear(32, 64), nn.GELU(),
            nn.Dropout(0.2), nn.Linear(64, 32),
        )
        self.vector_branch = nn.Sequential(
            nn.Linear(n_vectors * 32, 64),
            nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.2),
        )
        self.head = nn.Sequential(
            nn.Linear(256, 64), nn.GELU(), nn.Dropout(0.3), nn.Linear(64, 1)
        )
        with torch.no_grad():
            self.head[-1].bias.fill_(float(np.log(positive_rate / (1 - positive_rate))))
        self.attention = (
            nn.MultiheadAttention(32, 4, dropout=0.2, batch_first=True)
            if attention else None
        )
        self.attention_norm = nn.LayerNorm(32) if attention else None
        self.attention_dropout = nn.Dropout(0.2)

    def forward(self, numeric, categorical):
        if self.training:
            categorical = categorical.masked_fill(
                torch.rand_like(categorical, dtype=torch.float32) < 0.3, 0
            )
        embedded = torch.cat(
            [
                layer(categorical[:, index])
                for index, layer in enumerate(self.embeddings)
            ],
            dim=1,
        )
        vectors = numeric[:, self.n_dense :].reshape(-1, self.n_vectors, 34)
        projected = F.gelu(
            torch.einsum("bfi,fij->bfj", vectors[:, :, :33], self.vector_weights)
            + self.vector_bias
        )
        tokens = torch.where(
            vectors[:, :, 33:].bool(), projected, self.missing_tokens.unsqueeze(0)
        ) + self.field_tokens
        if self.attention is not None:
            normalized = self.attention_norm(tokens)
            interactions, _ = self.attention(
                normalized, normalized, normalized, need_weights=False
            )
            tokens = tokens + self.attention_dropout(interactions)
        tokens = tokens + self.vector_transform(tokens)
        fused = torch.cat(
            [
                self.scalar_branch(numeric[:, : self.n_dense]),
                self.category_branch(embedded),
                self.vector_branch(tokens.flatten(1)),
            ],
            dim=1,
        )
        return self.head(fused)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--rows", type=int, default=150_000)
    parser.add_argument("--data-dir", type=Path, default=ROOT / "data")
    parser.add_argument("--sample-path", type=Path)
    parser.add_argument("--epochs", type=int, default=30)
    parser.add_argument("--patience", type=int, default=5)
    parser.add_argument("--batch-size", type=int, default=1024)
    parser.add_argument("--min-category-count", type=int, default=10)
    parser.add_argument("--variant", choices=["both", "control", "attention"], default="both")
    parser.add_argument("--device", choices=["auto", "cpu", "mps", "cuda"], default="auto")
    parser.add_argument("--threads", type=int, default=4)
    parser.add_argument("--output-dir", type=Path, default=ROOT / "artifacts/local-custom-v1")
    args = parser.parse_args()
    if min(
        args.rows, args.epochs, args.patience, args.batch_size,
        args.min_category_count, args.threads,
    ) <= 0:
        parser.error("All numerical arguments must be positive")
    device = args.device
    if device == "auto":
        device = (
            "cuda" if torch.cuda.is_available()
            else "mps" if torch.backends.mps.is_available()
            else "cpu"
        )
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but unavailable")
    if device == "mps" and not torch.backends.mps.is_available():
        raise RuntimeError("Apple MPS requested but unavailable")
    torch.set_num_threads(args.threads)
    started = time.monotonic()
    sample_hash = None
    if args.sample_path:
        sample = pl.read_parquet(args.sample_path)
        if sample.height != args.rows:
            raise ValueError("Cached sample row count does not match --rows")
        with args.sample_path.open("rb") as stream:
            sample_hash = hashlib.file_digest(stream, "sha256").hexdigest()
    else:
        sample = load_sample(
            args.data_dir / "train.parquet", args.rows, 42, expand_embeddings=True
        )
    sample = sample.sort("_time")
    if sample.height < 2 or sample["_time"].null_count():
        raise ValueError("Chronological validation requires nonmissing timestamps")
    cutoff = sample["_time"][int(sample.height * 0.8)]
    split = int(sample["_time"].search_sorted(cutoff, side="left"))
    train = sample.slice(0, split)
    valid = sample.slice(split)
    for label, frame in [("training", train), ("validation", valid)]:
        if set(frame[TARGET].unique().to_list()) != {0, 1}:
            raise ValueError(f"{label} must contain both binary target classes")
    x_train, c_train, x_valid, c_valid, state = prepare_custom_features(
        train, valid, args.min_category_count
    )
    y_train = torch.tensor(train[TARGET].to_numpy().copy(), dtype=torch.float32)
    y_valid = valid[TARGET].to_numpy().copy()
    report = {
        "seed": 42,
        "sample_rows": sample.height,
        "sample_sha256": sample_hash,
        "training_rows": train.height,
        "validation_rows": valid.height,
        "cutoff": str(cutoff),
        "input_features": sample.width - 2,
        "dense_channels_after_preprocessing": state["n_dense"],
        "vector_fields": state["vector_fields"],
        "original_cardinalities": state["original_cardinalities"],
        "retained_cardinalities": state["cardinalities"],
        "min_category_count": args.min_category_count,
        "validation_unknown_or_rare_category_rates": state["validation_unknown_or_rare_category_rates"],
        "missing_vectors_train": state["missing_vectors_train"],
        "missing_vectors_valid": state["missing_vectors_valid"],
        "device": device,
        "threads": args.threads,
        "max_epochs": args.epochs,
        "patience": args.patience,
        "batch_size": args.batch_size,
        "optimizer": {"name": "AdamW", "learning_rate": 0.0005, "weight_decay": 0.01},
        "versions": {
            name: version(name)
            for name in ["torch", "numpy", "polars", "scikit-learn"]
        },
        "competition_submission": False,
        "results": [],
    }
    positive_rate = float(y_train.mean().item())
    del sample, train, valid
    gc.collect()
    output = args.output_dir
    output.mkdir(parents=True, exist_ok=True)
    joblib.dump(state, output / "preprocessing.joblib")
    report["preparation_seconds"] = time.monotonic() - started
    variants = ["control", "attention"] if args.variant == "both" else [args.variant]
    for variant in variants:
        torch.manual_seed(42)
        model_config = {
            "n_dense": state["n_dense"],
            "cardinalities": state["cardinalities"],
            "n_vectors": len(state["vector_fields"]),
            "attention": variant == "attention",
            "positive_rate": positive_rate,
        }
        model = RecommendationNetwork(**model_config).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.01)
        variant_output = output / variant
        variant_output.mkdir(exist_ok=True)
        parameters = sum(parameter.numel() for parameter in model.parameters())
        print(
            f"Variant: {variant}; device: {device}; train: {len(y_train):,}; "
            f"valid: {len(y_valid):,}; parameters: {parameters:,}; cutoff: {cutoff}",
            flush=True,
        )
        fitted = fit_model(
            model, model_config, optimizer, x_train, c_train, y_train,
            x_valid, c_valid, y_valid, device=device, output=variant_output,
            epochs=args.epochs, patience=args.patience, batch_size=args.batch_size,
        )
        result = {
            "variant": variant,
            "parameters": parameters,
            "model_config": model_config,
            **fitted,
        }
        report["results"].append(result)
        (output / "results.json").write_text(json.dumps(report, indent=2) + "\n")
        print(
            f"{variant}: AUC={fitted['validation_auc']:.6f}; "
            f"epoch={fitted['best_epoch']}; training={fitted['training_seconds']:.1f}s",
            flush=True,
        )
        del model, optimizer
        gc.collect()
        if device == "mps":
            torch.mps.empty_cache()
        elif device == "cuda":
            torch.cuda.empty_cache()
    print("Requested validation results saved; no competition submission performed", flush=True)




In [ ]:
import argparse
from datetime import datetime
import hashlib
from importlib.metadata import version
import json
from pathlib import Path
import time

import joblib
import numpy as np
import polars as pl
import pyarrow.parquet as pq
from sklearn.metrics import roc_auc_score
import torch



def digest(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model-dir", type=Path, required=True)
    parser.add_argument("--data-dir", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--batch-size", type=int, default=1024)
    parser.add_argument("--threads", type=int, default=4)
    parser.add_argument("--device", choices=["cpu", "mps", "cuda"], default="cuda")
    args = parser.parse_args()
    if min(args.batch_size, args.threads) <= 0:
        parser.error("Batch size and threads must be positive")
    if args.device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA inference requested but unavailable")
    if args.device == "mps" and not torch.backends.mps.is_available():
        raise RuntimeError("Apple MPS inference requested but unavailable")
    started = time.monotonic()
    torch.set_num_threads(args.threads)
    training = json.loads((args.model_dir / "results.json").read_text())
    variants = [entry for entry in training["results"] if entry["variant"] == "attention"]
    if len(variants) != 1:
        raise ValueError("Expected one recorded attention checkpoint")
    reference = variants[0]
    versions = {name: version(name) for name in training["versions"]}
    if versions != training["versions"]:
        raise RuntimeError(
            "Inference package versions differ from training; use the original Kaggle "
            f"environment. Expected {training['versions']}; got {versions}"
        )
    checkpoint_path = args.model_dir / "attention/best_model.pt"
    preprocessor_path = args.model_dir / "preprocessing.joblib"
    state = joblib.load(preprocessor_path)
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    if checkpoint["model_config"] != reference["model_config"]:
        raise ValueError("Checkpoint model configuration differs from the training record")
    model = RecommendationNetwork(**checkpoint["model_config"]).to(args.device)
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    model.eval()
    print(
        f"Inference device: {args.device}; unchanged best epoch: {reference['best_epoch']}; "
        f"training rows: {training['training_rows']:,}; no retraining",
        flush=True,
    )

    def probabilities(raw):
        numeric, categorical = transform_custom_features(
            features(raw, expand_embeddings=True), state
        )
        return predict(model, numeric, categorical, args.device, args.batch_size)

    cutoff = datetime.fromisoformat(training["cutoff"])
    labels, validation_predictions = [], []
    for batch in pq.ParquetFile(args.data_dir / "train.parquet").iter_batches(
        batch_size=8 * args.batch_size
    ):
        raw = pl.from_arrow(batch).filter(pl.col("feature1") >= cutoff)
        if raw.is_empty():
            continue
        labels.append(raw[TARGET].to_numpy())
        validation_predictions.append(probabilities(raw))
    if not labels:
        raise ValueError("No validation rows found for the recorded timestamp cutoff")
    labels = np.concatenate(labels)
    validation_predictions = np.concatenate(validation_predictions)
    if len(labels) != training["validation_rows"]:
        raise ValueError("Replayed validation row count differs from the training record")
    validation_auc = float(roc_auc_score(labels, validation_predictions))
    if abs(validation_auc - reference["validation_auc"]) > 0.000005:
        raise ValueError(
            f"Replayed AUC {validation_auc} differs from checkpoint AUC "
            f"{reference['validation_auc']}"
        )
    print(f"Replayed validation ROC AUC: {validation_auc:.9f}", flush=True)

    template = pl.read_csv(args.data_dir / "sample_submission.csv")
    if template.columns != ["id", TARGET] or template["id"].n_unique() != template.height:
        raise ValueError("Unexpected submission columns or duplicate template IDs")
    test = pq.ParquetFile(args.data_dir / "test.parquet")
    if test.metadata.num_rows != template.height:
        raise ValueError("Test and template row counts differ")
    predictions, offset = [], 0
    for batch in test.iter_batches(batch_size=8 * args.batch_size):
        raw = pl.from_arrow(batch)
        if "id" not in raw.columns:
            raise ValueError("Test data is missing submission IDs")
        if raw["id"].to_list() != template["id"].slice(offset, raw.height).to_list():
            raise ValueError("Test IDs do not match sample submission order")
        predictions.append(probabilities(raw))
        offset += raw.height
        if offset % (25 * 8 * args.batch_size) == 0:
            print(f"Predicted {offset:,}/{template.height:,} test rows", flush=True)
    predictions = np.concatenate(predictions)
    if len(predictions) != template.height or not np.isfinite(predictions).all():
        raise ValueError("Invalid prediction count or nonfinite probabilities")
    if not ((predictions >= 0) & (predictions <= 1)).all():
        raise ValueError("Probabilities must be in [0, 1]")
    args.output_dir.mkdir(parents=True, exist_ok=True)
    destination = args.output_dir / "submission.csv"
    template.with_columns(pl.Series(TARGET, predictions)).write_csv(destination)
    report = {
        "checkpoint_best_epoch": reference["best_epoch"],
        "training_rows": training["training_rows"],
        "validation_rows": len(labels),
        "training_validation_auc": reference["validation_auc"],
        "replayed_validation_auc": validation_auc,
        "test_rows": len(predictions),
        "prediction_min": float(predictions.min()),
        "prediction_mean": float(predictions.mean()),
        "prediction_max": float(predictions.max()),
        "checkpoint_sha256": digest(checkpoint_path),
        "preprocessor_sha256": digest(preprocessor_path),
        "training_results_sha256": digest(args.model_dir / "results.json"),
        "submission_sha256": digest(destination),
        "submission_bytes": destination.stat().st_size,
        "device": args.device,
        "batch_size": args.batch_size,
        "versions": versions,
        "inference_seconds": time.monotonic() - started,
        "refit": False,
        "automatic_competition_submission": False,
    }
    (args.output_dir / "inference-results.json").write_text(
        json.dumps(report, indent=2) + "\n"
    )
    print(
        f"Saved {len(predictions):,} predictions; no retraining or automatic submission",
        flush=True,
    )




In [ ]:
main()


In [ ]:
print((OUTPUT / "inference-results.json").read_text())
